# 02 - Preprocessing Pipeline

This notebook demonstrates:
- Missing value handling strategies
- Feature engineering (lag features, rolling stats, seasonal encodings)
- SMOGN-based oversampling for extreme events
- Data normalization and temporal splitting

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import CAMELSIndDataLoader
from src.data.preprocessor import StreamflowPreprocessor
from src.features.engineer import FeatureEngineer
from src.utils.helpers import load_config

%matplotlib inline

In [ ]:
config = load_config('../configs/config.yaml')

loader = CAMELSIndDataLoader('../data/raw')
if not loader.list_catchments():
    loader.generate_sample_data(n_catchments=5, n_days=2000)

data = loader.load_all_catchments()
print(f'Raw data shape: {data.shape}')
print(f'Columns: {list(data.columns)}')

In [ ]:
# Feature engineering
engineer = FeatureEngineer(config.get('features', {}))
engineered = engineer.transform(data)
feature_names = engineer.get_feature_names(engineered)

print(f'Engineered data shape: {engineered.shape}')
print(f'Number of features: {len(feature_names)}')
print(f'Feature names: {feature_names[:20]}...')

In [ ]:
# Preprocessing
preprocessor = StreamflowPreprocessor(config.get('preprocessing', {}))
X_train, X_test, y_train, y_test = preprocessor.fit_transform(engineered)

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}, range: [{y_train.min():.2f}, {y_train.max():.2f}]')
print(f'y_test: {y_test.shape}, range: [{y_test.min():.2f}, {y_test.max():.2f}]')

In [ ]:
# Visualize target distribution before/after SMOGN
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(y_train, bins=50, color='steelblue', alpha=0.7)
axes[0].set_title('Training Target Distribution')
axes[0].set_xlabel('Streamflow (mm/day)')

axes[1].hist(y_test, bins=50, color='coral', alpha=0.7)
axes[1].set_title('Test Target Distribution')
axes[1].set_xlabel('Streamflow (mm/day)')

plt.tight_layout()
plt.show()